### Task 7 – Understanding User Preferences and Preparing Interaction Data

In this task, you will understand how user preferences and interactions can be 
represented as data. You will prepare an interaction dataset that will be used in our 
upcoming collaborative filtering model. 

### 1. Understand user interaction data 

Learn the meaning and purpose of the following concepts: 

 User ID 

 TED Talk ID 

 Ratings 

 Likes and dislikes 

 Watch history 

 Saved or bookmarked talks 

 User preferences 

 User-item interaction matrix 

Understand how these interactions help a recommendation system identify user 
interests. 

### 2. Design a user interaction dataset 
You must create your own sample dataset with multiple users and TED Talks. If actual user interaction data is unavailable, you may create simulated data for this learning exercise. Clearly mention that it is simulated data.

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Define the dataset path

# Load the cleaned TED Talks dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Display the first five records
df.head()

,Unnamed: 0,talk_id,title,speaker_1,duration,topics,url,description,transcript,title_clean,speaker_1_clean,topics_clean,description_clean,transcript_clean,combined_text
0,0,1,Averting the climate crisis,Al Gore,977,"['alternative energy', 'cars', 'climate change...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,al gore,alternative energy cars climate change culture...,with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre...",averting the climate crisis alternative energy...
1,1,92,The best stats you've ever seen,Hans Rosling,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you've ever seen,hans rosling,africa asia google demo economics global issue...,you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac...",the best stats you've ever seen africa asia go...
2,2,7,Simplicity sells,David Pogue,1286,"['computers', 'entertainment', 'interface desi...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,david pogue,computers entertainment interface design media...,new york times columnist david pogue takes aim...,"(music: ""the sound of silence,"" simon & garfun...",simplicity sells computers entertainment inter...
3,3,53,Greening the ghetto,Majora Carter,1116,"['MacArthur grant', 'activism', 'business', 'c...",https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today â€” and I'm very happy th...,greening the ghetto,majora carter,macarthur grant activism business cities envir...,"in an emotionally charged talk, macarthur-winn...",if you're here today â€” and i'm very happy th...,greening the ghetto macarthur grant activism b...
4,4,66,Do schools kill creativity?,Sir Ken Robinson,1164,"['children', 'creativity', 'culture', 'dance',...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity?,sir ken robinson,children creativity culture dance education pa...,sir ken robinson makes an entertaining and pro...,good morning. how are you? (audience) good. it...,do schools kill creativity? children creativit...


In [3]:
df = df.drop(columns="Unnamed: 0",errors="ignore")

In [4]:
# Display dataset dimensions
print("Dataset Shape:", df.shape)

# Display column names
print("\nDataset Columns:")
print(df.columns.tolist())

# Check unique TED Talk IDs
print("\nUnique TED Talks:", df["talk_id"].nunique())

# Check missing and duplicate talk IDs
print("Missing Talk IDs:", df["talk_id"].isna().sum())
print("Duplicate Talk IDs:", df["talk_id"].duplicated().sum())

Dataset Shape: (4005, 14)

Dataset Columns:
['talk_id', 'title', 'speaker_1', 'duration', 'topics', 'url', 'description', 'transcript', 'title_clean', 'speaker_1_clean', 'topics_clean', 'description_clean', 'transcript_clean', 'combined_text']

Unique TED Talks: 4005
Missing Talk IDs: 0
Duplicate Talk IDs: 0


In [6]:
df["topics"].head()

0    ['alternative energy', 'cars', 'climate change...
1    ['Africa', 'Asia', 'Google', 'demo', 'economic...
2    ['computers', 'entertainment', 'interface desi...
3    ['MacArthur grant', 'activism', 'business', 'c...
4    ['children', 'creativity', 'culture', 'dance',...
Name: topics, dtype: str

In [5]:
# Define user-interest categories

import ast

# Define five user-interest categories
topic_groups = {
    "Science": {
        "science", "biology", "physics", "chemistry",
        "astronomy", "climate change", "environment",
        "space", "nature"
    },

    "Technology": {
        "technology", "computers", "software", "internet",
        "engineering", "innovation", "artificial intelligence",
        "robots", "data"
    },

    "Education": {
        "education", "teaching", "learning", "schools",
        "children", "student", "literacy"
    },

    "Arts & Creativity": {
        "music", "art", "design", "creativity", "film",
        "movies", "dance", "performance", "photography"
    },

    "Health & Psychology": {
        "health", "psychology", "brain", "happiness",
        "mental health", "medicine", "well-being",
        "neuroscience"
    }
}

# Convert topics from string representation into Python lists
df["topics_list"] = df["topics"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

display(df["topics_list"])
print("Interest categories created successfully.")
print("Total categories:", len(topic_groups))

0       [alternative energy, cars, climate change, cul...
1       [Africa, Asia, Google, demo, economics, global...
2       [computers, entertainment, interface design, m...
3       [MacArthur grant, activism, business, cities, ...
4       [children, creativity, culture, dance, educati...
                              ...                        
4000    [activism, data, technology, mental health, so...
4001    [TED-Ed, education, history, animation, intell...
4002         [society, law, policy, justice system, TEDx]
4003    [TED-Ed, education, animation, United States, ...
4004    [animals, TED-Ed, animation, oceans, science, ...
Name: topics_list, Length: 4005, dtype: object

Interest categories created successfully.
Total categories: 5


In [10]:
df["topics_list"].isnull().sum()

np.int64(0)

In [7]:
# Assign categories and select 50 talks

def assign_category(topics):
    talk_topics = {
        str(topic).strip().casefold()
        for topic in topics
    }

    for category, keywords in topic_groups.items():
        if talk_topics.intersection(keywords):
            return category

    return None


df["interest_category"] = df["topics_list"].apply(assign_category)

display(df["interest_category"])

0                   Science
1       Health & Psychology
2                Technology
3                   Science
4                 Education
               ...         
4000             Technology
4001              Education
4002                    NaN
4003              Education
4004                Science
Name: interest_category, Length: 4005, dtype: str

In [8]:
df["interest_category"].isnull().sum()

np.int64(848)

In [13]:
# Select 10 talks from each category
selected_talks = []

for category in topic_groups.keys():

    category_talks = df[
        df["interest_category"] == category
    ]

    sampled_talks = category_talks.sample(
        n=10,
        random_state=42
    )

    selected_talks.append(sampled_talks)

print(len(selected_talks))

5


In [14]:
display(selected_talks)

[      talk_id                                              title  \
 3883    23794  How do cancer cells behave differently from he...   
 949      1203                  How language transformed humanity   
 3262    24436                 The myth behind the Chinese zodiac   
 1201     1467              Will our kids be a different species?   
 3762    54394                  Why I protest for climate justice   
 2846    18928  The story of 'Oumuamua, the first visitor from...   
 689       901                    Don't build your home, grow it!   
 2864    19829            What is the coldest thing in the world?   
 834      1080             Saving faces: A facial surgeon's craft   
 1892     2220  How to revive a neighborhood: with imagination...   
 
                speaker_1  duration  \
 3883      Charles Morton       219   
 949           Mark Pagel      1210   
 3262       Megan Campisi       242   
 1201       Juan Enriquez      1008   
 3762          Jane Fonda       855   
 2846

In [15]:
# Combine all selected talks
talk_pool = pd.concat(
    selected_talks,
    ignore_index=True
)

# Keep only the required columns
talk_pool = talk_pool[
    ["talk_id", "title", "speaker_1", "topics", "interest_category"]
]

print("Selected TED Talks:", len(talk_pool))
print("\nTalks per category:")
print(talk_pool["interest_category"].value_counts())

display(talk_pool.head(10))

Selected TED Talks: 50

Talks per category:
interest_category
Science                10
Technology             10
Education              10
Arts & Creativity      10
Health & Psychology    10
Name: count, dtype: int64


,talk_id,title,speaker_1,topics,interest_category
0,23794,How do cancer cells behave differently from he...,Charles Morton,"['animation', 'TED-Ed', 'human body', 'disease...",Science
1,1203,How language transformed humanity,Mark Pagel,"['biology', 'communication', 'culture', 'evolu...",Science
2,24436,The myth behind the Chinese zodiac,Megan Campisi,"['history', 'culture', 'world cultures', 'anci...",Science
3,1467,Will our kids be a different species?,Juan Enriquez,"['TEDx', 'brain', 'culture', 'evolution', 'gen...",Science
4,54394,Why I protest for climate justice,Jane Fonda,"['climate change', 'environment', 'activism', ...",Science
5,18928,"The story of 'Oumuamua, the first visitor from...",Karen J. Meech,"['astronomy', 'extraterrestrial life', 'explor...",Science
6,901,"Don't build your home, grow it!",Mitchell Joachim,"['TED Fellows', 'architecture', 'design', 'env...",Science
7,19829,What is the coldest thing in the world?,Lina Marieth Hoyos,"['TED-Ed', 'science', 'physics']",Science
8,1080,Saving faces: A facial surgeon's craft,Iain Hutchison,"['body language', 'community', 'culture', 'des...",Science
9,2220,How to revive a neighborhood: with imagination...,Theaster Gates,"['architecture', 'art', 'cities', 'design', 'e...",Science


In [16]:
# Create user profiles
user_profiles = {}

user_id = 1

for category in topic_groups.keys():

    for _ in range(4):

        user_profiles[f"U{user_id:03d}"] = category

        user_id += 1

user_profiles

{'U001': 'Science',
 'U002': 'Science',
 'U003': 'Science',
 'U004': 'Science',
 'U005': 'Technology',
 'U006': 'Technology',
 'U007': 'Technology',
 'U008': 'Technology',
 'U009': 'Education',
 'U010': 'Education',
 'U011': 'Education',
 'U012': 'Education',
 'U013': 'Arts & Creativity',
 'U014': 'Arts & Creativity',
 'U015': 'Arts & Creativity',
 'U016': 'Arts & Creativity',
 'U017': 'Health & Psychology',
 'U018': 'Health & Psychology',
 'U019': 'Health & Psychology',
 'U020': 'Health & Psychology'}

In [17]:
# Convert user profiles into a DataFrame
users_df = pd.DataFrame(
    list(user_profiles.items()),
    columns=["user_id", "preferred_category"]
)

display(users_df)

print("Total simulated users:", users_df["user_id"].nunique())

,user_id,preferred_category
0,U001,Science
1,U002,Science
2,U003,Science
3,U004,Science
4,U005,Technology
5,U006,Technology
6,U007,Technology
7,U008,Technology
8,U009,Education
9,U010,Education


Total simulated users: 20


In [18]:
# Generate simulated interactions

# Set random seed for reproducibility
np.random.seed(42)

interaction_records = []

# Generate interactions for each user
for user_id, preferred_category in user_profiles.items():

    # Select 14 talks for the current user
    user_talks = talk_pool.sample(
        n=14,
        random_state=np.random.randint(0, 100000)
    )

    for _, talk in user_talks.iterrows():

        # Generate ratings according to user preferences
        if talk["interest_category"] == preferred_category:
            rating = np.random.choice([4, 5])
        else:
            rating = np.random.choice([1, 2, 3])

        # Derive like status from the rating
        liked = rating >= 4

        # Generate bookmark status
        bookmarked = np.random.choice(
            [True, False],
            p=[0.20, 0.80]
        )

        interaction_records.append({
            "user_id": user_id,
            "talk_id": talk["talk_id"],
            "rating": rating,
            "liked": liked,
            "watched": True,
            "bookmarked": bookmarked
        })


# Create the interaction DataFrame
interactions_df = pd.DataFrame(interaction_records)

# Display sample interactions
display(interactions_df.head(10))

print("Total interactions:", len(interactions_df))
print("Unique users:", interactions_df["user_id"].nunique())
print("Unique TED Talks:", interactions_df["talk_id"].nunique())

,user_id,talk_id,rating,liked,watched,bookmarked
0,U001,13589,1,False,True,True
1,U001,1618,1,False,True,False
2,U001,46462,2,False,True,True
3,U001,54394,4,True,True,False
4,U001,32267,1,False,True,False
5,U001,24330,3,False,True,True
6,U001,24138,2,False,True,False
7,U001,901,5,True,True,False
8,U001,9048,1,False,True,False
9,U001,1689,2,False,True,False


Total interactions: 280
Unique users: 20
Unique TED Talks: 50


In [19]:
# Validate the interaction data

# Check missing values
print("Missing values:")
print(interactions_df.isnull().sum())

# Check duplicate user-talk interactions
print(
    "\nDuplicate user-talk pairs:",
    interactions_df.duplicated(
        subset=["user_id", "talk_id"]
    ).sum()
)

# Verify that every talk ID exists in the cleaned dataset
valid_talk_ids = set(df["talk_id"])

invalid_talk_ids = set(interactions_df["talk_id"]) - valid_talk_ids

print("\nInvalid TED Talk IDs:", len(invalid_talk_ids))

# Check rating range
print("\nRating range:")
print(interactions_df["rating"].min(), "-", interactions_df["rating"].max())

Missing values:
user_id       0
talk_id       0
rating        0
liked         0
watched       0
bookmarked    0
dtype: int64

Duplicate user-talk pairs: 0

Invalid TED Talk IDs: 0

Rating range:
1 - 5


In [46]:
# Save the interaction dataset

# Create interaction directory if it does not exist
os.makedirs("../data/interaction", exist_ok=True)

# Define output path
INTERACTION_PATH = "../data/interaction/user_interactions.csv"

# Save simulated interaction data
interactions_df.to_csv(
    INTERACTION_PATH,
    index=False
)

print("Interaction dataset saved successfully!")
print("File location:", INTERACTION_PATH)

Interaction dataset saved successfully!
File location: ../data/interaction/user_interactions.csv


In [20]:
interactions_df.head(20)

,user_id,talk_id,rating,liked,watched,bookmarked
0,U001,13589,1,False,True,True
1,U001,1618,1,False,True,False
2,U001,46462,2,False,True,True
3,U001,54394,4,True,True,False
4,U001,32267,1,False,True,False
5,U001,24330,3,False,True,True
6,U001,24138,2,False,True,False
7,U001,901,5,True,True,False
8,U001,9048,1,False,True,False
9,U001,1689,2,False,True,False


### 3. Understand the User-Item Interaction Matrix 

Convert your interaction data into a user-item matrix. 

Understand: 

 Rows represent users. 

 Columns represent TED Talks. 

 Values represent user ratings. 

 Missing values indicate that a user has not rated or interacted with that talk.

In [21]:
# Create the user-item interaction matrix

# Convert interaction data into a user-item matrix
user_item_matrix = interactions_df.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

# Display the matrix
display(user_item_matrix)

talk_id,64,149,205,328,501,529,589,865,883,901,...,24491,27951,32267,43362,46462,46906,48856,54394,58119,62627
user_id,,,,,,,,,,,,,,,,,,,,,
U001,1.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,5.0,...,3.0,NaN,1.0,NaN,2.0,NaN,NaN,4.0,NaN,NaN
U002,NaN,2.0,1.0,NaN,2.0,NaN,NaN,NaN,NaN,4.0,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
U003,2.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,5.0,...,NaN,3.0,NaN,NaN,2.0,NaN,NaN,4.0,NaN,1.0
U004,NaN,3.0,NaN,NaN,NaN,2.0,NaN,NaN,3.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN
U005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,2.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,3.0,2.0
U006,3.0,NaN,4.0,NaN,2.0,NaN,5.0,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,3.0,1.0,NaN,NaN,NaN
U007,NaN,NaN,NaN,2.0,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,2.0,1.0,NaN,NaN,NaN
U008,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,2.0,NaN,...,NaN,2.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
U009,2.0,NaN,NaN,4.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Inspect the matrix dimensions

print("User-Item Matrix Shape:", user_item_matrix.shape)

print("\nTotal Users:", user_item_matrix.shape[0])
print("Total TED Talks:", user_item_matrix.shape[1])

print("\nMissing Ratings:", user_item_matrix.isna().sum().sum())

User-Item Matrix Shape: (20, 50)

Total Users: 20
Total TED Talks: 50

Missing Ratings: 720


In [23]:
# Calculate rating coverage

# Calculate total possible user-talk combinations
total_possible_ratings = (
    user_item_matrix.shape[0] * user_item_matrix.shape[1]
)

# Count actual ratings
actual_ratings = user_item_matrix.count().sum()

# Calculate missing ratings
missing_ratings = user_item_matrix.isna().sum().sum()

# Calculate matrix density
matrix_density = (
    actual_ratings / total_possible_ratings
) * 100

print("Total Possible Ratings:", total_possible_ratings)
print("Actual Ratings:", actual_ratings)
print("Missing Ratings:", missing_ratings)
print("Matrix Density:", round(matrix_density, 2), "%")

Total Possible Ratings: 1000
Actual Ratings: 280
Missing Ratings: 720
Matrix Density: 28.0 %


In [51]:
# Export the user-item matrix

# Save the interaction matrix
MATRIX_PATH = "../data/interaction/user_item_matrix.csv"

user_item_matrix.to_csv(MATRIX_PATH)

print("User-item matrix saved successfully!")
print("File location:", MATRIX_PATH)

User-item matrix saved successfully!
File location: ../data/interaction/user_item_matrix.csv


### 4. Perform basic interaction data analysis 

Answer the following: 

 How many unique users are present? 

 How many unique TED Talks are included? 

 Which TED Talks received the highest ratings? 

 Which users have rated the most talks? 

 Which talks have received the most interactions? 

 What does a missing rating indicate? 

In [24]:
# Find TED Talks with the highest ratings --> We'll calculate average ratings for each TED Talk.

# Calculate average ratings and number of ratings per talk
talk_rating_analysis = (
    interactions_df.groupby("talk_id")
    .agg(
        average_rating=("rating", "mean"),
        total_ratings=("rating", "count")
    )
    .sort_values(
        by=["average_rating", "total_ratings"],
        ascending=[False, False]
    )
    .reset_index()
)

# Add actual TED Talk titles
talk_rating_analysis = talk_rating_analysis.merge(
    talk_pool[["talk_id", "title"]],
    on="talk_id",
    how="left"
)

display(talk_rating_analysis.head(10))

,talk_id,average_rating,total_ratings,title
0,865,3.600000,5,Bring on the learning revolution!
1,901,3.571429,7,"Don't build your home, grow it!"
2,1167,3.500000,6,Break the silence for suicide attempt survivors
3,2026,3.500000,2,What's next in 3D printing
4,27951,3.428571,7,What makes a superhero?
5,24491,3.333333,6,Why do people join cults?
6,328,3.166667,6,Dog-friendly dog training
7,1689,3.125000,8,Your brain is more than a bag of chemicals
8,589,3.000000,5,17 words of architectural inspiration
9,24436,3.000000,5,The myth behind the Chinese zodiac


In [25]:
# Find users who rated the most talks

user_rating_analysis = (
    interactions_df.groupby("user_id")
    .agg(
        total_rated_talks=("talk_id", "nunique"),
        average_rating=("rating", "mean")
    )
    .sort_values(
        by="total_rated_talks",
        ascending=False
    )
)

display(user_rating_analysis)

,total_rated_talks,average_rating
user_id,,
U001,14,2.285714
U002,14,2.428571
U003,14,2.428571
U004,14,3.000000
U005,14,2.500000
U006,14,2.285714
U007,14,1.928571
U008,14,2.500000
U009,14,2.357143


In [55]:
# Find talks with the most interactions

talk_interaction_counts = (
    interactions_df.groupby("talk_id")
    .size()
    .reset_index(name="total_interactions")
    .sort_values(
        by="total_interactions",
        ascending=False
    )
)

talk_interaction_counts = talk_interaction_counts.merge(
    talk_pool[["talk_id", "title"]],
    on="talk_id",
    how="left"
)

display(talk_interaction_counts.head(10))

,talk_id,total_interactions,title
0,1108,8,The genius puppetry behind War Horse
1,1203,8,How language transformed humanity
2,2580,8,The new American Dream
3,1689,8,Your brain is more than a bag of chemicals
4,901,7,"Don't build your home, grow it!"
5,149,7,How to get (a new) hip
6,9048,7,"The search for ""aha!"" moments"
7,2264,7,Gorgeous portraits of the world's vanishing pe...
8,1077,7,"The future of business is the ""mesh"""
9,21977,7,"Why should you read ""One Hundred Years of Soli..."


### 5. Understand the purpose of collaborative filtering 

Study the basic idea of collaborative filtering. Understand how a system can recommend talks based on the preferences of users with similar interests. 

For example: 

If User A and User B have rated several TED Talks similarly, the system may identify that they have similar interests and recommend a talk that User A liked but User B has not watched. 

### Understanding Collaborative Filtering

Collaborative Filtering is a recommendation technique that uses the preferences and interactions of multiple users to recommend items.

Unlike Content-Based Filtering, which recommends items based on their features and content, Collaborative Filtering identifies patterns in user-item interactions.

**Types of Collaborative Filtering:**

1. **User-Based Collaborative Filtering:** Identifies users with similar preferences and recommends items that similar users have rated highly but the target user has not rated yet.

2. **Item-Based Collaborative Filtering:** Identifies items that receive similar rating patterns from users and recommends items based on the target user's previously liked or highly rated items.

**Example:**

Suppose User A and User B have both rated several TED Talks similarly. User B has also given a high rating to another TED Talk that User A has not watched or rated.

A user-based collaborative filtering system may recommend that TED Talk to User A because User B has demonstrated similar interests.

**Observations:**

- The user-item matrix represents interactions between users and TED Talks.
- Missing ratings indicate that a user has not rated a particular TED Talk.
- Multiple users interacting with the same TED Talks provide opportunities to identify similarities.
- Collaborative Filtering can discover user preferences from interaction patterns without depending entirely on TED Talk content.

**Note:** The interaction dataset used in this activity is simulated for educational purposes and does not represent actual TED user activity.